# Predicting Without a Pipeline

The deployment counterpart to `titanic-without-using-pipeline.ipynb` — three separately-pickled objects (`ohe_sex`, `ohe_embarked`, `clf`) are loaded back and applied **by hand, in the exact same order** used during training, to turn one raw input into a prediction.

## 1. Load the Three Saved Pieces

Unlike the pipeline version (a single `pipe.pkl`), every transformer that was fit during training has to be loaded back **separately** here — and applied in precisely the right order, since nothing enforces that order automatically the way a `Pipeline` does.

In [1]:
import pickle
import numpy as np

In [2]:
ohe_sex = pickle.load(open('models/ohe_sex.pkl','rb'))
ohe_embarked = pickle.load(open('models/ohe_embarked.pkl','rb'))
clf = pickle.load(open('models/clf.pkl','rb'))

FileNotFoundError: [Errno 2] No such file or directory: 'models/ohe_sex.pkl'

## 2. The Raw Input

`Pclass, Sex, Age, SibSp, Parch, Fare, Embarked` — same 7 raw values as the pipeline version's input, but here *none* of them are usable by the model yet; each one needs the same manual treatment applied at training time.

In [ ]:
# Assume user input
# Pclass/gender/age/SibSp/Parch/Fare/Embarked
test_input = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'],dtype=object).reshape(1,7)

In [ ]:
test_input

## 3. Encoding `Sex`

Column index `1` (`Sex`) goes through the *same fitted* `ohe_sex` encoder used in training — `.transform()`, not `.fit_transform()`, since it must reuse the categories already learned.

In [ ]:
test_input_sex = ohe_sex.transform(test_input[:,1].reshape(1,1))

In [ ]:
test_input_sex

## 4. Encoding `Embarked`

Same idea, for the last column (`Embarked`, index `-1`), using the separately-fitted `ohe_embarked` encoder.

In [ ]:
test_input_embarked = ohe_embarked.transform(test_input[:,-1].reshape(1,1))

In [ ]:
test_input_embarked

## 5. Extracting `Age`

`Age` (index `2`) needs no encoding — it's already numeric — just pulled out and reshaped to line up with the other pieces for concatenation.

In [ ]:
test_input_age = test_input[:,2].reshape(1,1)

## 6. Reassembling in the Exact Training Order

`test_input[:,[0,3,4,5]]` grabs `Pclass, SibSp, Parch, Fare` — the columns that needed no preprocessing at all — by their exact original positions. The final concatenation order (remaining columns → age → sex → embarked) **must match** the order used when `X_train_transformed` was built during training, or the model will silently receive columns in the wrong positions and produce meaningless predictions with no error at all.

In [ ]:
test_input_transformed = np.concatenate((test_input[:,[0,3,4,5]],test_input_age,test_input_sex,test_input_embarked),axis=1)

In [ ]:
test_input_transformed.shape

## 7. Predicting

In [ ]:
clf.predict(test_input_transformed)

### Getting a Probability — `predict_proba()`

Same addition as the pipeline version — the class label alone hides how confident the model actually is.

In [ ]:
probabilities = clf.predict_proba(test_input_transformed)
print(f"P(did not survive) = {probabilities[0][0]:.2%}")
print(f"P(survived)        = {probabilities[0][1]:.2%}")


## Summary

| Task | Method |
|---|---|
| Load each saved transformer/model separately | `pickle.load(open('file.pkl', 'rb'))` — one call per object |
| Apply a fitted encoder to new data | `.transform()` *(never `.fit_transform()` here)* |
| Reassemble in training's exact column order | `np.concatenate((...), axis=1)` |
| Get prediction confidence | `clf.predict_proba(X)` |

### The cost of skipping a pipeline
This entire notebook exists to do what `predict-using-pipeline.ipynb` does in two lines (`pipe = pickle.load(...)`, `pipe.predict(raw_input)`). Every step here depends on remembering the *exact* column order and transformer used at training time — get one index wrong, or load the wrong `.pkl` file, and the model will still run and still produce an answer, just a silently wrong one.